# 32-SiPM kNN Localization Baseline

This notebook builds a simple baseline for single-muon localization using the 32 SiPM count vector per event. It parses the known scan positions from `macros/muon_scan.mac`, builds one mean detector-response template per run/location, and predicts a new event's `(x, z)` position using weighted k-nearest neighbors in SiPM-response space.

Target to beat: current basic SiPM weighting gives about `2 cm` 1-sigma single-hit deviation.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "macros").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAINING_DIR = PROJECT_ROOT / "analysis/training_scan_data_1024"
TEST_DIR = PROJECT_ROOT / "analysis/test_scan_data_1000"
TEST_MANIFEST = TEST_DIR / "test_manifest.csv"
MACRO_PATH = PROJECT_ROOT / "macros/muon_scan.mac"
PREDICTIONS_CSV = PROJECT_ROOT / "analysis/test_scan_data_1000_knn_predictions.csv"
SUMMARY_CSV = PROJECT_ROOT / "analysis/test_scan_data_1000_knn_summary.csv"
ERROR_PNG = PROJECT_ROOT / "analysis/test_scan_data_1000_knn_error_map.png"

RUN_FILE_RE = re.compile(r"MUON-run(?P<run>\d+)_sipm_counts_by_event\.csv$")
SIPM_COLUMNS = [f"sipm_{i}" for i in range(100, 116)] + [f"sipm_{i}" for i in range(300, 316)]

K = 8
NORMALIZE = "l1"      # options: "none", "l1", "sqrt_l1"
METRIC = "cosine"     # good first choices: "cosine" or "euclidean"
EPS = 1e-12

## Load Labels and Training Events

In [ ]:
def parse_muon_scan_positions(macro_path: Path) -> pd.DataFrame:
    """Return run_id -> known GPS position from /gps/position lines."""
    rows = []
    for line in macro_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) == 5 and parts[0] == "/gps/position":
            x_cm, y_cm, z_cm = map(float, parts[1:4])
            rows.append({
                "run_id": len(rows),
                "x_cm": x_cm,
                "y_cm": y_cm,
                "z_cm": z_cm,
                "x_mm": 10.0 * x_cm,
                "z_mm": 10.0 * z_cm,
            })
    labels = pd.DataFrame(rows)
    if labels.empty:
        raise ValueError(f"No /gps/position lines found in {macro_path}")
    return labels


def load_training_events(training_dir: Path) -> pd.DataFrame:
    frames = []
    for path in sorted(training_dir.glob("MUON-run*_sipm_counts_by_event.csv")):
        match = RUN_FILE_RE.match(path.name)
        if not match:
            continue
        run_id = int(match.group("run"))
        df = pd.read_csv(path)
        missing = [col for col in ["EventID", *SIPM_COLUMNS] if col not in df.columns]
        if missing:
            raise KeyError(f"{path} is missing columns: {missing}")
        df = df[["EventID", *SIPM_COLUMNS]].copy()
        df.insert(0, "run_id", run_id)
        df = df.rename(columns={"EventID": "event_id"})
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No per-run count CSVs found in {training_dir}")
    return pd.concat(frames, ignore_index=True)


labels = parse_muon_scan_positions(MACRO_PATH)
events = load_training_events(TRAINING_DIR)
events_labeled = events.merge(labels, on="run_id", how="left", validate="many_to_one")

print(f"labels: {len(labels)} scan positions")
print(f"events: {len(events_labeled)} single-muon events")
events_labeled.head()

## Build Run-Averaged Templates

Each scan point has 100 single-muon events. The baseline compares a new single event against one mean 32-vector template per scan point.

In [ ]:
def make_run_templates(events_df: pd.DataFrame, labels_df: pd.DataFrame) -> pd.DataFrame:
    templates = events_df.groupby("run_id", as_index=False)[SIPM_COLUMNS].mean()
    templates = templates.merge(labels_df, on="run_id", how="left", validate="one_to_one")
    return templates.sort_values("run_id").reset_index(drop=True)


templates = make_run_templates(events, labels)
templates.head()

## Normalization and kNN Helpers

In [ ]:
def vectorize(df: pd.DataFrame) -> np.ndarray:
    return df[SIPM_COLUMNS].to_numpy(dtype=np.float64)


def normalize_vectors(x: np.ndarray, mode: str = "l1") -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    if mode == "none":
        return x
    if mode == "l1":
        return x / np.maximum(x.sum(axis=1, keepdims=True), EPS)
    if mode == "sqrt_l1":
        y = np.sqrt(np.maximum(x, 0.0))
        return y / np.maximum(y.sum(axis=1, keepdims=True), EPS)
    raise ValueError(f"Unknown normalization mode: {mode}")


def fit_template_knn(templates_df: pd.DataFrame, k: int = K, metric: str = METRIC, normalize: str = NORMALIZE):
    x_template = normalize_vectors(vectorize(templates_df), normalize)
    model = NearestNeighbors(n_neighbors=k, metric=metric)
    model.fit(x_template)
    return {"model": model, "templates": templates_df.reset_index(drop=True), "normalize": normalize, "metric": metric, "k": k}


def predict_positions(events_df: pd.DataFrame, knn_state: dict, weight_power: float = 1.0) -> pd.DataFrame:
    x_event = normalize_vectors(vectorize(events_df), knn_state["normalize"])
    distances, indices = knn_state["model"].kneighbors(x_event)
    templates_df = knn_state["templates"]

    pred_rows = []
    for row_idx, (row_distances, row_indices) in enumerate(zip(distances, indices)):
        neighbor_rows = templates_df.iloc[row_indices]
        weights = 1.0 / np.maximum(row_distances, EPS) ** weight_power
        weights = weights / weights.sum()

        pred_x_cm = float(np.dot(weights, neighbor_rows["x_cm"].to_numpy()))
        pred_z_cm = float(np.dot(weights, neighbor_rows["z_cm"].to_numpy()))
        nearest = neighbor_rows.iloc[0]

        pred_rows.append({
            "row_index": events_df.index[row_idx],
            "pred_x_cm": pred_x_cm,
            "pred_z_cm": pred_z_cm,
            "nearest_run_id": int(nearest["run_id"]),
            "nearest_x_cm": float(nearest["x_cm"]),
            "nearest_z_cm": float(nearest["z_cm"]),
            "nearest_distance": float(row_distances[0]),
            "neighbor_run_ids": tuple(int(v) for v in neighbor_rows["run_id"]),
            "neighbor_distances": tuple(float(v) for v in row_distances),
        })
    return pd.DataFrame(pred_rows).set_index("row_index")


knn_state = fit_template_knn(templates, k=K, metric=METRIC, normalize=NORMALIZE)

## Predict Random-Simulation Test Events

This uses the prepared `analysis/test_scan_data_1000` folder. Each clean CSV is one random single-muon event, and `test_manifest.csv` carries the known random source position.

In [ ]:
def load_test_events(test_dir: Path, manifest_path: Path) -> pd.DataFrame:
    manifest = pd.read_csv(manifest_path)
    rows = []
    for _, meta in manifest.sort_values("event_id").iterrows():
        path = test_dir / meta["output_file"]
        df = pd.read_csv(path)
        if len(df) != 1:
            raise ValueError(f"Expected one row in {path}, found {len(df)}")
        missing = [col for col in ["EventID", *SIPM_COLUMNS] if col not in df.columns]
        if missing:
            raise KeyError(f"{path} is missing columns: {missing}")
        row = df.iloc[0].to_dict()
        row["event_id"] = int(meta["event_id"])
        row["raw_file"] = meta["raw_file"]
        row["true_x_cm"] = float(meta["true_x_cm"])
        row["true_z_cm"] = float(meta["true_z_cm"])
        rows.append(row)
    return pd.DataFrame(rows).sort_values("event_id").reset_index(drop=True)


test_events = load_test_events(TEST_DIR, TEST_MANIFEST)
test_predictions = predict_positions(test_events, knn_state)
test_results = pd.concat([test_events.reset_index(drop=True), test_predictions.reset_index(drop=True)], axis=1)
test_results.head()

## Evaluation Helpers

Use these once the 1000 random events have truth positions. Errors are reported in cm so the current `2 cm` baseline is directly comparable.

In [ ]:
def attach_error_columns(results: pd.DataFrame, truth_prefix: str = "true") -> pd.DataFrame:
    out = results.copy()
    x_col = f"{truth_prefix}_x_cm"
    z_col = f"{truth_prefix}_z_cm"
    if x_col not in out.columns and "x_cm" in out.columns:
        x_col = "x_cm"
    if z_col not in out.columns and "z_cm" in out.columns:
        z_col = "z_cm"
    if x_col not in out.columns or z_col not in out.columns:
        raise KeyError("Need truth columns true_x_cm/true_z_cm or x_cm/z_cm")

    out["err_x_cm"] = out["pred_x_cm"] - out[x_col]
    out["err_z_cm"] = out["pred_z_cm"] - out[z_col]
    out["err_r_cm"] = np.hypot(out["err_x_cm"], out["err_z_cm"])
    return out


def summarize_errors(results_with_errors: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n_events": len(results_with_errors),
        "mean_err_x_cm": results_with_errors["err_x_cm"].mean(),
        "sigma_err_x_cm": results_with_errors["err_x_cm"].std(ddof=1),
        "mean_err_z_cm": results_with_errors["err_z_cm"].mean(),
        "sigma_err_z_cm": results_with_errors["err_z_cm"].std(ddof=1),
        "median_err_r_cm": results_with_errors["err_r_cm"].median(),
        "p68_err_r_cm": results_with_errors["err_r_cm"].quantile(0.68),
        "p95_err_r_cm": results_with_errors["err_r_cm"].quantile(0.95),
    })


def plot_error_map(results_with_errors: pd.DataFrame, truth_prefix: str = "true"):
    x_col = f"{truth_prefix}_x_cm" if f"{truth_prefix}_x_cm" in results_with_errors.columns else "x_cm"
    z_col = f"{truth_prefix}_z_cm" if f"{truth_prefix}_z_cm" in results_with_errors.columns else "z_cm"
    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(results_with_errors[x_col], results_with_errors[z_col], c=results_with_errors["err_r_cm"], s=18, cmap="viridis")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("true x [cm]")
    ax.set_ylabel("true z [cm]")
    ax.set_title("kNN radial error by true hit position")
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("radial error [cm]")
    return fig, ax


test_results_with_errors = attach_error_columns(test_results)
summary = summarize_errors(test_results_with_errors)
test_results_with_errors.to_csv(PREDICTIONS_CSV, index=False)
summary.to_frame("value").to_csv(SUMMARY_CSV)
fig, ax = plot_error_map(test_results_with_errors)
fig.savefig(ERROR_PNG, dpi=180)
summary

## Optional: Internal Sanity Check

This is not the final benchmark because these events come from the scan itself. It is useful for tuning `K`, `NORMALIZE`, and `METRIC`, but the real test is the 1000 random-position simulation.

In [ ]:
def evaluate_scan_events_against_templates(events_df: pd.DataFrame, labels_df: pd.DataFrame, templates_df: pd.DataFrame) -> pd.DataFrame:
    predictions = predict_positions(events_df, fit_template_knn(templates_df, k=K, metric=METRIC, normalize=NORMALIZE))
    results = pd.concat([events_df.reset_index(drop=True), predictions.reset_index(drop=True)], axis=1)
    results = results.merge(labels_df[["run_id", "x_cm", "z_cm"]], on="run_id", how="left", validate="many_to_one")
    return attach_error_columns(results, truth_prefix="")


# scan_check = evaluate_scan_events_against_templates(events, labels, templates)
# summarize_errors(scan_check)